# Module 5 - Class 3: PCA on Mall Customers

**Dataset:** Mall Customer Segmentation  
**Objective:** Use PCA to reduce dimensions, visualize customer segments, and interpret principal components.

### What you will learn
- How PCA works (eigenvalue decomposition of covariance matrix)
- Scree plot and cumulative explained variance
- Reducing to 2D for visualization
- Interpreting PCA loadings in business terms

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 1. Load Mall Customers Dataset

In [ ]:
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/Mall_Customers.csv"
df = pd.read_csv(url)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Select numeric features for PCA
# Encode Gender as 0/1
df['Gender_encoded'] = (df['Gender'] == 'Male').astype(int)

features = ['Gender_encoded', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].copy()

print("Features for PCA:")
print(X.describe().round(2))

## 2. Standardize Features

PCA is sensitive to feature scale — features with larger ranges will dominate the principal components.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("After scaling (mean ~ 0, std ~ 1):")
print(pd.DataFrame(X_scaled, columns=features).describe().round(2))

## 3. PCA with All Components

In [ ]:
pca_full = PCA(n_components=4)
X_pca_full = pca_full.fit_transform(X_scaled)

print("Explained Variance Ratio per Component:")
for i, var in enumerate(pca_full.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.4f} ({var*100:.1f}%)")

print(f"\nTotal explained variance: {pca_full.explained_variance_ratio_.sum()*100:.1f}%")

## 4. Scree Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

components = range(1, 5)
var_ratio = pca_full.explained_variance_ratio_
cumulative = np.cumsum(var_ratio)

# Scree plot
axes[0].bar(components, var_ratio, color='#3498db', alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].set_xticks(list(components))
axes[0].set_xticklabels([f'PC{i}' for i in components])
for i, v in enumerate(var_ratio):
    axes[0].text(i+1, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# Cumulative explained variance
axes[1].plot(components, cumulative, 'o-', color='#e74c3c', linewidth=2, markersize=8)
axes[1].axhline(y=0.9, color='gray', linestyle='--', label='90% threshold')
axes[1].fill_between(components, cumulative, alpha=0.1, color='#e74c3c')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xticks(list(components))
axes[1].legend()
for i, v in enumerate(cumulative):
    axes[1].text(i+1, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Reduce to 2 Components and Scatter Plot

In [ ]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
pca_df['Gender'] = df['Gender'].values

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Gender', alpha=0.7, s=60)
plt.title(f'Mall Customers in PCA Space (2D)\nExplained: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%', fontsize=13)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. PCA Loadings

Loadings tell us how much each original feature contributes to each principal component.

In [ ]:
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f'PC{i+1}' for i in range(4)],
    index=features
)

print("PCA Loadings (how original features map to components):")
print(loadings.round(4))
print()

# Heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(loadings, annot=True, fmt='.3f', cmap='RdBu_r', center=0, linewidths=0.5)
plt.title('PCA Loadings Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Biplot: vectors showing feature directions in PC1-PC2 space
plt.figure(figsize=(10, 7))

# Scatter points
plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], alpha=0.3, s=30, color='gray')

# Feature vectors
scale = 3  # Scale arrows for visibility
for i, feat in enumerate(features):
    plt.arrow(0, 0,
              loadings.iloc[i, 0] * scale,
              loadings.iloc[i, 1] * scale,
              head_width=0.08, head_length=0.05, fc='red', ec='red', linewidth=2)
    plt.text(loadings.iloc[i, 0] * scale * 1.15,
             loadings.iloc[i, 1] * scale * 1.15,
             feat, fontsize=10, fontweight='bold', color='red')

plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA Biplot — Feature Directions', fontsize=13)
plt.axhline(y=0, color='black', linewidth=0.5)
plt.axvline(x=0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. TODO: Interpret the Principal Components

Look at the loadings table and biplot above, then answer:

1. **PC1 interpretation:** Which original features contribute most to PC1? What does PC1 represent in business terms? (e.g., does it separate rich vs poor customers? young vs old?)

2. **PC2 interpretation:** Which features dominate PC2? What business meaning does this axis carry?

3. How many components would you keep if you wanted to retain at least 90% of the variance?

**TODO: Your interpretation here**

*Write your answer in this cell.*


---
## Summary

| Concept | Details |
|---------|--------|
| PCA | Projects data onto directions of maximum variance |
| Standardization | Required before PCA — features must be on the same scale |
| Scree Plot | Shows variance per component — look for the "elbow" |
| Cumulative Variance | Helps decide how many components to keep (e.g., 90% threshold) |
| Loadings | Coefficients mapping original features to principal components |
| Biplot | Overlays feature vectors on the PCA scatter — shows feature directions |